## Setup Experiments


In [ ]:
!pip install gymnasium[classic-control]
!pip install torch
!pip install moviepy
!pip install highway-env


In [5]:
import gymnasium as gym
import numpy as np
import random
import copy
import torch
from collections import deque
import sys
from pathlib import Path

# Add HighwayEnv to Python path
project_root = Path.cwd()
highway_env_path = project_root / "HighwayEnv"
if str(highway_env_path) not in sys.path:
    sys.path.insert(0, str(highway_env_path))

# Import highway_env to register environments
import highway_env  # noqa: F401


## Replay Buffer


In [6]:
class MultiStepReplayBuffer:
    """n-step return 전용 리플레이 버퍼 (PER 제거 버전)."""

    def __init__(
        self,
        obs_shape,
        action_dim,
        capacity,
        device,
        n_step=5,
        gamma=0.995,
    ):
        """
        n_step: multi-step TD 길이
        gamma: discount factor
        """
        self.capacity = capacity
        self.device = device

        self.n_step = n_step
        self.gamma = gamma

        obs_dtype = np.float32

        self.obses = np.empty((capacity, *obs_shape), dtype=obs_dtype)
        self.next_obses = np.empty((capacity, *obs_shape), dtype=obs_dtype)

        self.actions = np.empty((capacity, 1), dtype=np.int64)
        # multi-step return 을 reward 자리에 저장
        self.rewards = np.empty((capacity, 1), dtype=np.float32)
        self.not_dones = np.empty((capacity, 1), dtype=np.float32)
        self.not_dones_no_max = np.empty((capacity, 1), dtype=np.float32)

        self.idx = 0
        self.full = False

        # n-step transition 생성을 위한 버퍼
        self.n_step_buffer = deque(maxlen=n_step)

    def __len__(self):
        return self.capacity if self.full else self.idx

    def _get_n_step_info(self):
        """n-step return 과 마지막 next_obs / done 정보 계산."""
        R = 0.0
        discount = 1.0
        next_obs = None
        done = False
        done_no_max = False

        for (obs, action, reward, next_o, d, d_no_max) in self.n_step_buffer:
            R += discount * reward
            discount *= self.gamma
            next_obs = next_o
            done = d
            done_no_max = d_no_max
            if done:
                break

        first_obs, first_action, _, _, _, _ = self.n_step_buffer[0]
        return first_obs, first_action, R, next_obs, done, done_no_max

    def _add_single_transition(
        self,
        obs,
        action,
        reward,
        next_obs,
        done,
        done_no_max,
    ):
        """이미 n-step 이 계산된 transition 을 실제 버퍼에 저장."""
        np.copyto(self.obses[self.idx], obs)
        np.copyto(self.actions[self.idx], action)
        np.copyto(self.rewards[self.idx], reward)
        np.copyto(self.next_obses[self.idx], next_obs)
        np.copyto(self.not_dones[self.idx], float(not done))
        np.copyto(self.not_dones_no_max[self.idx], float(not done_no_max))

        self.idx = (self.idx + 1) % self.capacity
        self.full = self.full or self.idx == 0

    def add(
        self,
        obs,
        action,
        reward,
        next_obs,
        done,
        done_no_max,
    ):
        """1-step transition 을 받아 n-step return 으로 변환 후 저장."""
        self.n_step_buffer.append(
            (obs, action, reward, next_obs, done, done_no_max)
        )

        # n_step 만큼 쌓이면, 맨 앞에서 시작하는 n-step transition 하나 생성
        if len(self.n_step_buffer) == self.n_step:
            first_obs, first_action, R, n_next_obs, n_done, n_done_no_max = (
                self._get_n_step_info()
            )
            self._add_single_transition(
                first_obs,
                first_action,
                R,
                n_next_obs,
                n_done,
                n_done_no_max,
            )
            self.n_step_buffer.popleft()

        # 에피소드 끝이면 나머지 step 들에 대해서도 truncated n-step 추가
        if done:
            while len(self.n_step_buffer) > 0:
                first_obs, first_action, R, n_next_obs, n_done, n_done_no_max = (
                    self._get_n_step_info()
                )
                self._add_single_transition(
                    first_obs,
                    first_action,
                    R,
                    n_next_obs,
                    n_done,
                    n_done_no_max,
                )
                self.n_step_buffer.popleft()

    def sample(self, batch_size):
        """uniform 샘플링으로 미니배치 반환."""
        length = len(self)
        if length == 0:
            raise ValueError("Replay buffer is empty!")
        batch_size = min(length, batch_size)

        indexes = np.random.randint(0, length, size=batch_size)

        obses = torch.from_numpy(self.obses[indexes]).to(self.device)
        actions = torch.from_numpy(self.actions[indexes]).to(self.device)
        rewards = torch.from_numpy(self.rewards[indexes]).to(self.device)
        next_obses = torch.from_numpy(self.next_obses[indexes]).to(self.device)
        not_dones = torch.from_numpy(self.not_dones[indexes]).to(self.device)
        not_dones_no_max = torch.from_numpy(
            self.not_dones_no_max[indexes]
        ).to(self.device)

        return {
            "obses": obses,
            "actions": actions,
            "rewards": rewards,
            "next_obses": next_obses,
            "not_dones": not_dones,
            "not_dones_no_max": not_dones_no_max,
        }


## Q Network


In [7]:
import torch.nn as nn
import torch.nn.functional as F
import math


def scaled_dot_product_attention(
    query,
    key,
    value,
    mask=None,
    dropout=None,
):
    """Scaled Dot Product Attention 계산"""
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / np.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask, -1e9)

    attention_weights = F.softmax(scores, dim=-1)

    if dropout is not None:
        attention_weights = dropout(attention_weights)

    output = torch.matmul(attention_weights, value)
    return output, attention_weights


class EgoAttention(nn.Module):
    """Social Attention: Ego 중심의 Multi-head Attention"""

    def __init__(
        self,
        feature_size=64,
        heads=4,
        dropout_factor=0.0,
    ):
        super().__init__()
        self.feature_size = feature_size
        self.heads = heads
        self.dropout_factor = dropout_factor
        self.features_per_head = feature_size // heads

        assert (
            feature_size % heads == 0
        ), f"feature_size({feature_size})은 heads({heads})로 나누어떨어져야 합니다."

        self.query_ego = nn.Linear(feature_size, feature_size, bias=False)
        self.key_all = nn.Linear(feature_size, feature_size, bias=False)
        self.value_all = nn.Linear(feature_size, feature_size, bias=False)
        self.fc_out = nn.Linear(feature_size, feature_size, bias=False)
        self.dropout = nn.Dropout(dropout_factor)

    def forward(
        self,
        ego,
        others,
        mask=None,
    ):
        batch_size = ego.shape[0]

        all_entities = torch.cat([ego, others], dim=1)
        n_entities = all_entities.shape[1]

        query = self.query_ego(ego)
        key = self.key_all(all_entities)
        value = self.value_all(all_entities)

        query = query.view(
            batch_size, 1, self.heads, self.features_per_head
        ).transpose(1, 2)

        key = key.view(
            batch_size, n_entities, self.heads, self.features_per_head
        ).transpose(1, 2)

        value = value.view(
            batch_size, n_entities, self.heads, self.features_per_head
        ).transpose(1, 2)

        if mask is not None:
            mask = mask.view(batch_size, 1, 1, n_entities).expand(
                batch_size, self.heads, 1, n_entities
            )

        attn_output, attention_weights = scaled_dot_product_attention(
            query, key, value, mask=mask, dropout=self.dropout
        )

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, 1, self.feature_size)

        output = self.fc_out(attn_output)
        output = (output + ego) / 2
        output = output.squeeze(1)
        attention_weights = attention_weights.squeeze(2)

        return output, attention_weights


class NoisyLinear(nn.Module):
    """NoisyNet용 Linear layer (factorized Gaussian noise)."""

    def __init__(self, in_features, out_features, sigma_init=0.5):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        self.weight_mu = nn.Parameter(torch.empty(out_features, in_features))
        self.weight_sigma = nn.Parameter(torch.empty(out_features, in_features))

        self.bias_mu = nn.Parameter(torch.empty(out_features))
        self.bias_sigma = nn.Parameter(torch.empty(out_features))

        self.sigma_init = sigma_init
        self.reset_parameters()

    def reset_parameters(self):
        mu_range = 1.0 / math.sqrt(self.in_features)
        self.weight_mu.data.uniform_(-mu_range, mu_range)
        self.bias_mu.data.uniform_(-mu_range, mu_range)
        self.weight_sigma.data.fill_(self.sigma_init / math.sqrt(self.in_features))
        self.bias_sigma.data.fill_(self.sigma_init / math.sqrt(self.out_features))

    def _scale_noise(self, size):
        x = torch.randn(size, device=self.weight_mu.device, dtype=self.weight_mu.dtype)
        return x.sign().mul_(x.abs().sqrt_())

    def forward(self, x):
        if self.training:
            epsilon_in = self._scale_noise(self.in_features)
            epsilon_out = self._scale_noise(self.out_features)
            weight_epsilon = torch.outer(epsilon_out, epsilon_in)
            bias_epsilon = epsilon_out

            weight = self.weight_mu + self.weight_sigma * weight_epsilon
            bias = self.bias_mu + self.bias_sigma * bias_epsilon
        else:
            weight = self.weight_mu
            bias = self.bias_mu
        return F.linear(x, weight, bias)


class DuelingQNetwork(nn.Module):
    def __init__(
        self,
        obs_shape,
        action_dim,
        hidden_dim=256,
        hidden_depth=2,
        num_heads=4,
        noisy_std=0.5,
    ):
        super().__init__()

        if len(obs_shape) == 2:
            self.num_vehicles, self.feature_dim = obs_shape
            self.num_frames = 1
        else:
            raise ValueError(
                f"DuelingQNetwork는 (num_vehicles, feature_dim) 형태의 obs_shape을 기대합니다. obs_shape={obs_shape}"
            )

        self.action_dim = action_dim
        self.hidden_dim = hidden_dim
        self.hidden_depth = hidden_depth
        self.num_heads = num_heads

        self.entity_encoder = nn.Linear(self.feature_dim, hidden_dim)

        self.ego_attention = EgoAttention(
            feature_size=hidden_dim,
            heads=num_heads,
            dropout_factor=0.0,
        )

        post_layers = []
        in_dim = hidden_dim
        for _ in range(hidden_depth):
            post_layers.append(nn.ReLU())
            post_layers.append(nn.Linear(in_dim, hidden_dim))
            in_dim = hidden_dim
        self.post_fc = nn.Sequential(*post_layers)

        self.value_stream = nn.Sequential(
            NoisyLinear(hidden_dim, hidden_dim, sigma_init=noisy_std),
            nn.ReLU(),
            NoisyLinear(hidden_dim, 1, sigma_init=noisy_std),
        )

        self.advantage_stream = nn.Sequential(
            NoisyLinear(hidden_dim, hidden_dim, sigma_init=noisy_std),
            nn.ReLU(),
            NoisyLinear(hidden_dim, action_dim, sigma_init=noisy_std),
        )

    def forward(self, obs):
        if obs.dim() == 2:
            obs = obs.unsqueeze(0)
        elif obs.dim() != 3:
            raise ValueError(f"지원하지 않는 obs 차원: {obs.shape}")

        B, N, F = obs.shape
        assert N == self.num_vehicles and F == self.feature_dim

        x = self.entity_encoder(obs)

        presence = obs[:, :, 0]
        presence_mask = presence < 0.5

        ego = x[:, 0:1, :]
        others = x[:, 1:, :]

        full_presence_mask = presence_mask

        attn_out, attention_weights = self.ego_attention(ego, others, mask=full_presence_mask)

        h = self.post_fc(attn_out)

        value = self.value_stream(h)
        adv = self.advantage_stream(h)

        adv_mean = adv.mean(dim=1, keepdim=True)
        q_values = value + adv - adv_mean

        return q_values


## DQN Agent


In [8]:
import abc
import torch.optim as optim
from dataclasses import dataclass


@dataclass
class DQNConfig:
    gamma: float = 0.995
    lr: float = 1e-3
    batch_size: int = 64
    replay_buffer_capacity: int = 100_000

    target_update_interval: int = 1_000
    train_freq: int = 1
    warmup_steps: int = 1_000

    max_grad_norm: float = 10.0
    hidden_dim: int = 256
    hidden_depth: int = 2

    n_step: int = 5


class Agent(object):
    def reset(self):
        pass

    @abc.abstractmethod
    def train(self, training=True):
        pass

    @abc.abstractmethod
    def update(self, replay_buffer, step):
        pass

    @abc.abstractmethod
    def act(self, obs, sample=False):
        pass


class DQNAgent(Agent):
    def __init__(
        self,
        obs_shape,
        action_dim,
        device,
        cfg,
    ):
        super().__init__()

        self.obs_shape = obs_shape
        self.action_dim = action_dim
        self.device = torch.device(device)
        self.cfg = cfg

        self.q_network = DuelingQNetwork(
            obs_shape=obs_shape,
            action_dim=action_dim,
            hidden_dim=cfg.hidden_dim,
            hidden_depth=cfg.hidden_depth,
            num_heads=4,
            noisy_std=2.0,  # 강한 초기 탐험: σ_init = 2.0 / √256 = 0.125
        ).to(self.device)

        self.target_q_network = DuelingQNetwork(
            obs_shape=obs_shape,
            action_dim=action_dim,
            hidden_dim=cfg.hidden_dim,
            hidden_depth=cfg.hidden_depth,
            num_heads=4,
            noisy_std=2.0,  # 강한 초기 탐험: σ_init = 2.0 / √256 = 0.125
        ).to(self.device)

        self.target_q_network.load_state_dict(self.q_network.state_dict())

        self.optimizer = optim.Adam(self.q_network.parameters(), lr=cfg.lr)

        self._last_q_mean = 0.0
        self._last_target_q_mean = 0.0
        self._last_grad_norm = 0.0
        self._noise_stats = {}  # σ 통계 추적

    def train(self, training=True):
        self.q_network.train(training)
        self.target_q_network.train(False)

    def reset(self):
        pass

    def _obs_to_tensor(self, obs):
        if isinstance(obs, np.ndarray):
            obs_array = obs
        else:
            obs_array = np.array(obs, dtype=np.float32)

        obs_tensor = torch.from_numpy(obs_array).float().to(self.device)
        if obs_tensor.dim() == 2:
            obs_tensor = obs_tensor.unsqueeze(0)
        return obs_tensor

    def get_noise_statistics(self):
        """현재 NoisyLinear 레이어의 σ 값 통계 계산"""
        noise_stats = {}

        def _collect_sigma(module, prefix=""):
            if isinstance(module, NoisyLinear):
                sigma_mean = module.weight_sigma.data.mean().item()
                sigma_max = module.weight_sigma.data.max().item()
                sigma_min = module.weight_sigma.data.min().item()
                noise_stats[f"{prefix}_weight_sigma_mean"] = sigma_mean
                noise_stats[f"{prefix}_weight_sigma_max"] = sigma_max
                noise_stats[f"{prefix}_weight_sigma_min"] = sigma_min

            for name, child in module.named_children():
                _collect_sigma(child, f"{prefix}_{name}" if prefix else name)

        _collect_sigma(self.q_network)
        return noise_stats

    def act(self, obs, sample=False):
        # NoisyNet이 exploration을 담당하므로 항상 greedy 선택
        obs_tensor = self._obs_to_tensor(obs)

        with torch.no_grad():
            q_values = self.q_network(obs_tensor)
            action = int(torch.argmax(q_values, dim=-1).item())
        return action

    def act_greedy(self, obs):
        obs_tensor = self._obs_to_tensor(obs)
        with torch.no_grad():
            q_values = self.q_network(obs_tensor)
            action = int(torch.argmax(q_values, dim=-1).item())
        return action

    def update(self, replay_buffer, step):
        if step < self.cfg.warmup_steps:
            return None

        if step % self.cfg.train_freq != 0:
            return None

        sample = replay_buffer.sample(self.cfg.batch_size)

        loss = self._update_q_network(
            obs=sample["obses"],
            actions=sample["actions"],
            rewards=sample["rewards"],
            next_obs=sample["next_obses"],
            not_dones=sample["not_dones"],
            not_dones_no_max=sample["not_dones_no_max"],
            step=step,
        )

        return float(loss.item())

    def _update_q_network(
        self,
        obs,
        actions,
        rewards,
        next_obs,
        not_dones,
        not_dones_no_max,
        step,
    ):
        q_values = self.q_network(obs)
        q_values = q_values.gather(1, actions.long())

        with torch.no_grad():
            next_q_online = self.q_network(next_obs)
            next_actions = next_q_online.argmax(dim=1, keepdim=True)

            next_q_target = self.target_q_network(next_obs)
            next_q_values = next_q_target.gather(1, next_actions)

            gamma_n = self.cfg.gamma ** self.cfg.n_step
            target_q = rewards + gamma_n * not_dones_no_max * next_q_values

        loss = F.mse_loss(q_values, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        grad_norm = nn.utils.clip_grad_norm_(self.q_network.parameters(), self.cfg.max_grad_norm)
        self.optimizer.step()

        if step % self.cfg.target_update_interval == 0:
            self.target_q_network.load_state_dict(self.q_network.state_dict())

        self._last_q_mean = float(q_values.mean().item())
        self._last_target_q_mean = float(target_q.mean().item())
        self._last_grad_norm = float(grad_norm.item())

        # σ 통계 업데이트
        self._noise_stats = self.get_noise_statistics()

        return loss


## Environment Helper Functions


In [9]:
def make_env(env_id, render_mode=None):
    """Highway 계열 환경 생성."""
    kwargs = {}
    if render_mode is not None:
        kwargs["render_mode"] = render_mode

    env = gym.make(env_id, **kwargs)
    return env


def make_default_obs_preprocessor(env, num_frames=1):
    """관측을 float32로 변환하는 전처리 (frame stacking 없음)."""
    def _process(obs):
        if isinstance(obs, np.ndarray):
            obs_arr = obs.astype(np.float32)
        else:
            obs_arr = np.array(obs, dtype=np.float32)
        return obs_arr

    def reset():
        pass

    _process.reset = reset

    return _process


def get_obs_shape(env, num_frames=1):
    """에이전트/리플레이 버퍼에서 사용할 관찰 shape 계산."""
    space = env.observation_space

    if hasattr(space, "shape") and space.shape is not None:
        return tuple(space.shape)

    raise ValueError(f"Unsupported observation space type: {space}")


## Evaluation Function


In [10]:
# Environment
eval_env = make_env("merge-v0", render_mode="rgb_array")
obs_preprocessor = make_default_obs_preprocessor(eval_env)

# Testing
def run_evaluate(forward_agent):
    # 평가 모드로 전환 (noise sampling 비활성화)
    forward_agent.train(False)

    num_eval_episodes = 10
    average_episode_step, average_episode_reward = 0, 0

    for episode in range(num_eval_episodes):
        if hasattr(obs_preprocessor, "reset"):
            obs_preprocessor.reset()

        obs, info = eval_env.reset()
        proc_obs = obs_preprocessor(obs)
        episode_step, episode_reward, done = 0, 0, False

        while not done:
            action = forward_agent.act_greedy(proc_obs)
            next_obs, reward, terminated, truncated, info = eval_env.step(action)

            episode_step += 1
            episode_reward += reward
            done = terminated or truncated
            proc_obs = obs_preprocessor(next_obs)

        average_episode_step += episode_step
        average_episode_reward += episode_reward

    average_episode_step /= num_eval_episodes
    average_episode_reward /= num_eval_episodes

    # 평가 완료 후 훈련 모드로 복구
    forward_agent.train(True)

    return average_episode_step, average_episode_reward


## Training Function


In [ ]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Environment
env_id = "merge-v0" # HighwayEnv 환경 ID
env = make_env(env_id, render_mode=None)
obs_preprocessor = make_default_obs_preprocessor(env)
_obs, _info = env.reset()
proc_obs = obs_preprocessor(_obs)

# 모델 저장 경로 설정
import os
from pathlib import Path
env_name = env_id.replace("-v0", "")  # "roundabout-v0" -> "roundabout"
model_dir = Path("./models")
model_dir.mkdir(exist_ok=True)
best_model_path = model_dir / f"{env_id}-best.pt"
final_model_path = model_dir / f"{env_id}-final.pt"

# Warmup 설정
USE_WARMUP = True  # True: 웜업 데이터 수집 후 학습 시작, False: 바로 학습 시작
WARMUP_STEPS = 10_000  # 웜업 단계 수

# DQN Config
cfg = DQNConfig(
    gamma=0.995,
    lr=1e-3,
    batch_size=64,
    replay_buffer_capacity=100_000,
    target_update_interval=1_000,
    train_freq=1,
    warmup_steps=WARMUP_STEPS if USE_WARMUP else 0,
    max_grad_norm=10.0,
    hidden_dim=256,
    hidden_depth=2,
    n_step=5,
)

# DQN Agent
forward_agent = DQNAgent(
    obs_shape=proc_obs.shape,
    action_dim=env.action_space.n,
    device=device,
    cfg=cfg,
)

# Replay Buffer
forward_replay_buffer = MultiStepReplayBuffer(
    obs_shape=proc_obs.shape,
    action_dim=env.action_space.n,
    capacity=cfg.replay_buffer_capacity,
    device=device,
    n_step=cfg.n_step,
    gamma=cfg.gamma,
)

# Training
LOG_EPOCH_FREQ = int(1e1)
LOG_EVAL_FREQ = int(5e1)  # 평가는 50 epoch마다 실행 (훈련 logging과 분리)
num_epochs = int(1e5)
num_forward_steps = int(2e2)
num_warm_up_steps = cfg.warmup_steps if USE_WARMUP else 0
global_step = 0

# Early Stopping 설정
EARLY_STOPPING_PATIENCE = 20  # 20번 평가 동안 개선 없으면 멈춤
EARLY_STOPPING_THRESHOLD = 1e-3  # 최소 개선 임계값
best_eval_score = float('-inf')
patience_counter = 0

for epoch in range(num_epochs):
    if hasattr(obs_preprocessor, "reset"):
        obs_preprocessor.reset()

    obs, info = env.reset()
    proc_obs = obs_preprocessor(obs)
    episode_reward = 0.0

    for forward_step in range(num_forward_steps):
        action = forward_agent.act(proc_obs, sample=True)
        next_obs, reward, terminated, truncated, info = env.step(action)
        global_step += 1

        forward_done = terminated or truncated
        forward_done_no_max = terminated
        episode_reward += reward

        proc_next_obs = obs_preprocessor(next_obs)

        # 웜업 완료 후부터 학습 시작
        if (
            len(forward_replay_buffer) >= forward_agent.cfg.batch_size
            and global_step >= num_warm_up_steps
        ):
            forward_agent.update(forward_replay_buffer, global_step)

        forward_replay_buffer.add(proc_obs, action, reward, proc_next_obs, forward_done, forward_done_no_max)
        proc_obs = proc_next_obs.copy()

        if forward_done:
            break

    if epoch % LOG_EPOCH_FREQ == 0:
        warmup_status = f"WARMUP ({global_step}/{num_warm_up_steps})" if global_step < num_warm_up_steps else "TRAINING"
        print("================================================================================================")
        print("EPOCH: ", epoch, " GLOBAL STEP: ", global_step, " STATUS: ", warmup_status, " FORWARD STEP: ", forward_step, " FORWARD BUFFER SIZE: ", len(forward_replay_buffer))
        print("EPISODE REWARD: ", episode_reward)

        # σ 값 출력 (간결한 테이블 형식)
        noise_stats = forward_agent.get_noise_statistics()
        print("\n[σ STATISTICS]")
        print(f"{'Stream':<20} {'Mean':>10} {'Max':>10} {'Min':>10}")
        print("-" * 50)

        # 스트림별로 정리
        for stream_name in ["value_stream", "advantage_stream"]:
            for layer_idx in [0, 2]:  # NoisyLinear는 index 0과 2
                key_mean = f"{stream_name}_{layer_idx}_weight_sigma_mean"
                key_max = f"{stream_name}_{layer_idx}_weight_sigma_max"
                key_min = f"{stream_name}_{layer_idx}_weight_sigma_min"

                if key_mean in noise_stats:
                    label = f"{stream_name}[{layer_idx}]"
                    print(f"{label:<20} {noise_stats[key_mean]:>10.6f} {noise_stats[key_max]:>10.6f} {noise_stats[key_min]:>10.6f}")

        print("================================================================================================")

    # 웜업 완료 후부터 평가 시작
    if epoch % LOG_EVAL_FREQ == 0 and global_step >= num_warm_up_steps:
        avg_episode_step, avg_episode_reward = run_evaluate(forward_agent)

        print("================================================================================================")
        print("AVG. EVAL EPISODE STEP: ", avg_episode_step)
        print("AVG. EVAL EPISODE REWARD: ", avg_episode_reward)

        # Early Stopping 체크 (현재 평가 평균 기반)
        if avg_episode_reward > best_eval_score + EARLY_STOPPING_THRESHOLD:
            best_eval_score = avg_episode_reward
            patience_counter = 0
            # 최고 성능 모델 저장
            torch.save(forward_agent.q_network.state_dict(), best_model_path)
            print(f"[BEST MODEL] Saved to {best_model_path}")
            print(f"[EARLY STOPPING] Best score updated: {best_eval_score:.4f} (Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE})")
        else:
            patience_counter += 1
            print(f"[EARLY STOPPING] No improvement. Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")

            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"\n[EARLY STOPPING] Training stopped! Best eval score: {best_eval_score:.4f}")
                break

        print("================================================================================================")

# 학습 종료 후 최종 모델 저장
torch.save(forward_agent.q_network.state_dict(), final_model_path)
print(f"\n[FINAL MODEL] Saved to {final_model_path}")
print(f"Training completed! Best eval score: {best_eval_score:.4f}")


In [ ]:
# 저장된 모델 로드 및 평가
print("=" * 100)
print("모델 로드 및 평가 시작")
print("=" * 100)

# 모델 파일 경로 설정 (필요에 따라 수정)
model_path = final_model_path  # 또는 best_model_path 사용 가능
print(f"모델 경로: {model_path}")

# 모델이 존재하는지 확인
if not model_path.exists():
    print(f"경고: 모델 파일이 존재하지 않습니다: {model_path}")
    print("대신 best_model_path를 사용합니다.")
    model_path = best_model_path

if not model_path.exists():
    print(f"에러: 모델 파일이 존재하지 않습니다: {best_model_path}")
    print("먼저 모델을 학습시켜야 합니다.")
else:
    # 평가용 에이전트 생성
    eval_agent = DQNAgent(
        obs_shape=proc_obs.shape,
        action_dim=env.action_space.n,
        device=device,
        cfg=cfg,
    )

    # 모델 가중치 로드
    eval_agent.q_network.load_state_dict(torch.load(model_path, map_location=device))
    eval_agent.target_q_network.load_state_dict(eval_agent.q_network.state_dict())
    eval_agent.train(False)  # 평가 모드

    print(f"모델 로드 완료: {model_path}")
    print("=" * 100)

    # 평가 실행
    num_eval_episodes = 100
    episode_steps = []
    episode_rewards = []
    episode_success = []  # 충돌 없이 완료한지 여부
    episode_q_values = []  # 각 스텝의 Q-value 저장
    episode_actions = []  # 각 에피소드에서 선택한 액션 분포

    print(f"\n{num_eval_episodes}개 에피소드 평가 시작...\n")

    for episode in range(num_eval_episodes):
        if hasattr(obs_preprocessor, "reset"):
            obs_preprocessor.reset()

        obs, info = eval_env.reset()
        proc_obs = obs_preprocessor(obs)
        episode_step, episode_reward, done = 0, 0, False
        episode_q_list = []
        episode_action_list = []
        crashed = False

        while not done:
            # Q-value 계산
            obs_tensor = eval_agent._obs_to_tensor(proc_obs)
            with torch.no_grad():
                q_values = eval_agent.q_network(obs_tensor)
                q_max = q_values.max().item()
                q_mean = q_values.mean().item()
                q_std = q_values.std().item()
                episode_q_list.append({
                    'max': q_max,
                    'mean': q_mean,
                    'std': q_std,
                    'values': q_values.cpu().numpy().flatten().tolist()
                })

            action = eval_agent.act_greedy(proc_obs)
            episode_action_list.append(action)

            next_obs, reward, terminated, truncated, info = eval_env.step(action)

            episode_step += 1
            episode_reward += reward
            done = terminated or truncated

            # 충돌 감지 (일반적으로 Highway 환경에서 terminated가 True면 충돌)
            if terminated and episode_reward < 0:
                crashed = True

            proc_obs = obs_preprocessor(next_obs)

        episode_steps.append(episode_step)
        episode_rewards.append(episode_reward)
        episode_success.append(not crashed)
        episode_q_values.append(episode_q_list)
        episode_actions.append(episode_action_list)

        # 진행 상황 출력 (10 에피소드마다)
        if (episode + 1) % 10 == 0:
            current_avg_reward = np.mean(episode_rewards)
            current_avg_step = np.mean(episode_steps)
            success_rate = np.mean(episode_success) * 100
            print(f"진행: {episode + 1}/{num_eval_episodes} | "
                  f"평균 리워드: {current_avg_reward:.4f} | "
                  f"평균 스텝: {current_avg_step:.2f} | "
                  f"성공률: {success_rate:.1f}%")

    # 통계 계산
    episode_steps = np.array(episode_steps)
    episode_rewards = np.array(episode_rewards)
    episode_success = np.array(episode_success)

    # Q-value 통계
    all_q_max = []
    all_q_mean = []
    for ep_q in episode_q_values:
        for step_q in ep_q:
            all_q_max.append(step_q['max'])
            all_q_mean.append(step_q['mean'])

    # 액션 분포
    all_actions = []
    for ep_actions in episode_actions:
        all_actions.extend(ep_actions)
    action_counts = {}
    for action in all_actions:
        action_counts[action] = action_counts.get(action, 0) + 1
    total_actions = len(all_actions)
    action_distribution = {k: (v / total_actions * 100) for k, v in action_counts.items()}

    # 결과 출력
    print("\n" + "=" * 100)
    print("평가 결과 (100 에피소드)")
    print("=" * 100)

    print("\n[에피소드 스텝 통계]")
    print(f"  평균: {episode_steps.mean():.2f}")
    print(f"  표준편차: {episode_steps.std():.2f}")
    print(f"  최소: {episode_steps.min()}")
    print(f"  최대: {episode_steps.max()}")
    print(f"  중앙값: {np.median(episode_steps):.2f}")

    print("\n[에피소드 리워드 통계]")
    print(f"  평균: {episode_rewards.mean():.4f}")
    print(f"  표준편차: {episode_rewards.std():.4f}")
    print(f"  최소: {episode_rewards.min():.4f}")
    print(f"  최대: {episode_rewards.max():.4f}")
    print(f"  중앙값: {np.median(episode_rewards):.4f}")
    print(f"  총 리워드 합계: {episode_rewards.sum():.4f}")

    print("\n[성공률 통계]")
    success_rate = episode_success.mean() * 100
    crash_rate = (1 - episode_success.mean()) * 100
    print(f"  성공률 (충돌 없음): {success_rate:.2f}%")
    print(f"  충돌률: {crash_rate:.2f}%")
    print(f"  성공한 에피소드 수: {episode_success.sum()}/{num_eval_episodes}")

    print("\n[Q-value 통계]")
    if all_q_max:
        print(f"  Q-value 최대값 평균: {np.mean(all_q_max):.4f}")
        print(f"  Q-value 최대값 표준편차: {np.std(all_q_max):.4f}")
        print(f"  Q-value 평균값 평균: {np.mean(all_q_mean):.4f}")
        print(f"  Q-value 평균값 표준편차: {np.std(all_q_mean):.4f}")

    print("\n[액션 분포]")
    for action in sorted(action_distribution.keys()):
        print(f"  액션 {action}: {action_distribution[action]:.2f}%")

    print("\n[노이즈 통계 (NoisyLinear σ 값)]")
    noise_stats = eval_agent.get_noise_statistics()
    print(f"{'Stream':<20} {'Mean':>10} {'Max':>10} {'Min':>10}")
    print("-" * 50)
    for stream_name in ["value_stream", "advantage_stream"]:
        for layer_idx in [0, 2]:
            key_mean = f"{stream_name}_{layer_idx}_weight_sigma_mean"
            key_max = f"{stream_name}_{layer_idx}_weight_sigma_max"
            key_min = f"{stream_name}_{layer_idx}_weight_sigma_min"
            if key_mean in noise_stats:
                label = f"{stream_name}[{layer_idx}]"
                print(f"{label:<20} {noise_stats[key_mean]:>10.6f} {noise_stats[key_max]:>10.6f} {noise_stats[key_min]:>10.6f}")

    print("\n[상위 10개 에피소드]")
    top_indices = np.argsort(episode_rewards)[-10:][::-1]
    for i, idx in enumerate(top_indices, 1):
        print(f"  {i}. 에피소드 {idx}: 리워드={episode_rewards[idx]:.4f}, 스텝={episode_steps[idx]}, 성공={'예' if episode_success[idx] else '아니오'}")

    print("\n[하위 10개 에피소드]")
    bottom_indices = np.argsort(episode_rewards)[:10]
    for i, idx in enumerate(bottom_indices, 1):
        print(f"  {i}. 에피소드 {idx}: 리워드={episode_rewards[idx]:.4f}, 스텝={episode_steps[idx]}, 성공={'예' if episode_success[idx] else '아니오'}")

    print("\n" + "=" * 100)
    print("평가 완료!")
    print("=" * 100)
